# APS360 Progress Report: human-like chess move prediction by rating band
Run all cells top to bottom on a **GPU Colab runtime** (Runtime > Change runtime type > T4 GPU).
Total time is roughly 20-30 min with the default sizes. At the end it zips everything into
`progress_outputs.zip` (figures, `results.json`, `numbers.tex`). Send that zip back.

Pipeline: stream one Lichess monthly dump -> filter -> encode boards -> split by game ->
baselines (capture heuristic, linear model) -> CNN -> figures -> test on a *different month*.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-chess", "zstandard"], check=False)

In [ ]:
import os, io, re, json, time, random, shutil
import numpy as np
import chess, chess.pgn
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SYNTHETIC = os.environ.get("SYNTHETIC", "0") == "1"   # only used for local testing of the code
TRAIN_MONTH, TEST_MONTH = "2025-01", "2025-02"        # test month is never used for training or tuning
GAMES_PER_BAND = 6000 if not SYNTHETIC else 40
TEST_GAMES_PER_BAND = 800 if not SYNTHETIC else 15
POS_PER_GAME = 12
BAND_EDGES = [1100, 1300, 1500, 1700, 1900, 2100]
NB = 5                 # number of rating bands
MAXL = 128             # max legal moves stored per position
MIN_PLY, MIN_CLOCK = 10, 30
EPOCHS = 12 if not SYNTHETIC else 2
BATCH = 512
SEED = 0
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", device, "| synthetic:", SYNTHETIC)
BAND_NAMES = [f"{BAND_EDGES[i]}-{BAND_EDGES[i+1]}" for i in range(NB)]

## 1. Data: stream, filter, encode

In [ ]:
def lichess_lines(month):
    """Stream a Lichess monthly dump line by line. Closing the generator stops the download."""
    import requests, zstandard
    url = f"https://database.lichess.org/standard/lichess_db_standard_rated_{month}.pgn.zst"
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    try:
        reader = zstandard.ZstdDecompressor().stream_reader(r.raw)
        for line in io.TextIOWrapper(reader, encoding="utf-8"):
            yield line
    finally:
        r.close()

def synthetic_lines(seed, n_games=4000):
    """Random legal games with fake ratings/clocks. ONLY for testing that the code runs."""
    rng = random.Random(seed)
    for _ in range(n_games):
        g = chess.pgn.Game(); b = g.board(); node = g
        w = rng.randint(1000, 2200); bl = w + rng.randint(-250, 250)
        g.headers.update({"Event": "Rated Rapid game", "WhiteElo": str(w), "BlackElo": str(bl),
                          "Termination": "Normal", "Result": "1-0"})
        clock = [600.0, 600.0]
        for ply in range(rng.randint(30, 90)):
            moves = list(b.legal_moves)
            if not moves: break
            mv = rng.choice(moves); b.push(mv); node = node.add_variation(mv)
            c = ply % 2; clock[c] -= rng.uniform(1, 8); node.set_clock(max(clock[c], 0))
            if b.is_game_over(): break
        exp = chess.pgn.StringExporter(headers=True, variations=False, comments=True)
        for line in (g.accept(exp) + "\n\n").splitlines(keepends=True):
            yield line

def iter_games(lines):
    buf = []
    for line in lines:
        if line.startswith("[Event ") and buf:
            yield "".join(buf); buf = []
        buf.append(line)
    if buf: yield "".join(buf)

TAG = re.compile(r'\[(\w+) "(.*?)"\]')
def headers(txt):
    return dict(TAG.findall(txt.split("\n\n", 1)[0]))

def band_of(h):
    """Return rating band index 0..4 or None if the game is filtered out."""
    ev = h.get("Event", "")
    if "Rated" not in ev or "Rapid" not in ev: return None          # rated rapid only
    if h.get("Termination") != "Normal": return None                # no time forfeits / abandons
    if "BOT" in (h.get("WhiteTitle"), h.get("BlackTitle")): return None   # humans only
    try:
        w, b = int(h["WhiteElo"]), int(h["BlackElo"])
    except (KeyError, ValueError):
        return None
    if abs(w - b) > 200: return None                                # similar-strength players
    m = (w + b) / 2
    if m < BAND_EDGES[0] or m >= BAND_EDGES[-1]: return None
    return int((m - BAND_EDGES[0]) // 200)

PIECES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
def encode(board):
    """17 x 8 x 8 uint8. Board must be from the mover's view (mover = WHITE, moving up the rows)."""
    x = np.zeros((17, 8, 8), np.uint8)
    for sq, p in board.piece_map().items():
        x[PIECES.index(p.piece_type) + (0 if p.color == chess.WHITE else 6), sq // 8, sq % 8] = 1
    x[12] = board.has_kingside_castling_rights(chess.WHITE)
    x[13] = board.has_queenside_castling_rights(chess.WHITE)
    x[14] = board.has_kingside_castling_rights(chess.BLACK)
    x[15] = board.has_queenside_castling_rights(chess.BLACK)
    if board.ep_square is not None:
        x[16, board.ep_square // 8, board.ep_square % 8] = 1
    return x

def process_game(txt, rng):
    """Return list of (x, label, legal_idx_array, ply) sampled from one game."""
    game = chess.pgn.read_game(io.StringIO(txt))
    if game is None: return []
    moves, ok = [], []
    node = game; ply = 0
    while node.variations:
        node = node.variation(0)
        moves.append(node.move)
        clk = node.clock()   # mover's clock AFTER the move
        ok.append(ply >= MIN_PLY and clk is not None and clk >= MIN_CLOCK)
        ply += 1
    elig = [i for i, o in enumerate(ok) if o]
    if not elig: return []
    pick = set(rng.sample(elig, min(POS_PER_GAME, len(elig))))
    out = []; board = game.board()
    for i, mv in enumerate(moves):
        if i in pick:
            bd = board if board.turn == chess.WHITE else board.mirror()
            m = mv if board.turn == chess.WHITE else chess.Move(chess.square_mirror(mv.from_square),
                                                                chess.square_mirror(mv.to_square))
            legal = sorted({l.from_square * 64 + l.to_square for l in bd.legal_moves})
            if 2 <= len(legal) <= MAXL:      # drop forced moves
                out.append((encode(bd), m.from_square * 64 + m.to_square, np.array(legal, np.int16), i))
        board.push(mv)
    return out

def build_dataset(month, games_per_band, seed, max_scan=4_000_000):
    rng = random.Random(seed)
    src = synthetic_lines(seed) if SYNTHETIC else lichess_lines(month)
    quota = [0] * NB; X, Y, L, B, G, P = [], [], [], [], [], []
    scanned = 0; kept_games = 0; t0 = time.time(); stats = dict(scanned=0, kept=0)
    for txt in iter_games(src):
        scanned += 1
        b = band_of(headers(txt))
        if b is not None and quota[b] < games_per_band:
            rows = process_game(txt, rng)
            if rows:
                quota[b] += 1; kept_games += 1
                for (x, y, l, ply) in rows:
                    X.append(x); Y.append(y); B.append(b); G.append(kept_games); P.append(ply)
                    pad = np.full(MAXL, -1, np.int16); pad[:len(l)] = l; L.append(pad)
        if scanned % 50000 == 0:
            print(f"  scanned {scanned:,} | kept games per band {quota} | {time.time()-t0:.0f}s")
        if min(quota) >= games_per_band or scanned >= max_scan: break
    src.close()
    stats.update(scanned=scanned, kept=kept_games, games_per_band=quota)
    d = dict(X=np.stack(X), y=np.array(Y, np.int16), legal=np.stack(L), band=np.array(B, np.int8),
             gid=np.array(G, np.int32), ply=np.array(P, np.int16))
    print(f"{month}: scanned {scanned:,} games, kept {kept_games:,}, {len(d['y']):,} positions, {time.time()-t0:.0f}s")
    return d, stats

train_raw, train_stats = build_dataset(TRAIN_MONTH, GAMES_PER_BAND, SEED)
np.savez_compressed(f"{OUT}/dataset_train_month.npz", **train_raw)

## 2. Split by game and look at the cleaned data

In [ ]:
def split_by_game(d, seed=SEED):
    rng = np.random.RandomState(seed)
    gids = np.unique(d["gid"]); rng.shuffle(gids)
    n = len(gids); tr, va = gids[:int(.8 * n)], gids[int(.8 * n):int(.9 * n)]
    g = d["gid"]
    return (np.isin(g, tr), np.isin(g, va), ~np.isin(g, np.concatenate([tr, va])))

mtr, mva, mte = split_by_game(train_raw)
def take(d, m): return {k: v[m] for k, v in d.items()}
tr, va, te = take(train_raw, mtr), take(train_raw, mva), take(train_raw, mte)
print("positions train/val/test:", len(tr["y"]), len(va["y"]), len(te["y"]))
assert not (set(tr["gid"]) & set(te["gid"])) and not (set(tr["gid"]) & set(va["gid"]))   # no game leakage

pos_per_band = [int((train_raw["band"] == b).sum()) for b in range(NB)]
games_per_band = train_stats["games_per_band"]
avg_legal = float((train_raw["legal"] >= 0).sum(1).mean())
print("games per band:", games_per_band, "| positions per band:", pos_per_band, "| avg legal moves:", round(avg_legal, 1))

In [ ]:
GLYPH_OWN, GLYPH_OPP = "♙♘♗♖♕♔", "♟♞♝♜♛♚"
def draw_board(ax, x, moves=()):
    """x: 17x8x8 from mover's view (mover at bottom). moves: list of (idx, color, style)."""
    for r in range(8):
        for c in range(8):
            ax.add_patch(plt.Rectangle((c, r), 1, 1, color="#f0d9b5" if (r + c) % 2 == 1 else "#b58863"))
    for k in range(12):
        for r, c in zip(*np.nonzero(x[k])):
            g = GLYPH_OWN[k] if k < 6 else GLYPH_OPP[k - 6]
            ax.text(c + .5, r + .5, g, ha="center", va="center", fontsize=17, color="black")
    for idx, col, ls in moves:
        f, t = idx // 64, idx % 64
        ax.annotate("", xy=(t % 8 + .5, t // 8 + .5), xytext=(f % 8 + .5, f // 8 + .5),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=2.2, ls=ls, alpha=.85))
    ax.set_xlim(0, 8); ax.set_ylim(0, 8); ax.set_aspect("equal"); ax.axis("off")

def sq(i): return chess.square_name(i)
def mv_name(idx): return sq(idx // 64) + sq(idx % 64)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
ax[0].bar(range(NB), pos_per_band, color="#4c78a8")
ax[0].set_xticks(range(NB)); ax[0].set_xticklabels(BAND_NAMES, fontsize=7)
ax[0].set_title("Positions per rating band", fontsize=9); ax[0].set_ylabel("# positions", fontsize=8)
ax[1].hist(train_raw["ply"], bins=40, color="#f58518")
ax[1].set_title("Ply of sampled positions", fontsize=9); ax[1].set_xlabel("ply", fontsize=8)
i0 = 0
draw_board(ax[2], tr["X"][i0], [(int(tr["y"][i0]), "#2ca02c", "-")])
ax[2].set_title(f"Cleaned sample, band {BAND_NAMES[int(tr['band'][i0])]}\nplayed {mv_name(int(tr['y'][i0]))} (mover at bottom)", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_data.png", dpi=200); plt.close()

## 3. Models, training loop, evaluation

In [ ]:
def to_dev(d):
    return dict(X=torch.from_numpy(d["X"]).to(device), y=torch.from_numpy(d["y"].astype(np.int64)).to(device),
                legal=torch.from_numpy(d["legal"].astype(np.int64)).to(device),
                band=torch.from_numpy(d["band"].astype(np.int64)).to(device))
Dtr, Dva, Dte = to_dev(tr), to_dev(va), to_dev(te)

def make_input(Xb, bandb, cond=True):
    x = Xb.float()
    oh = F.one_hot(bandb, NB).float()[:, :, None, None].expand(-1, -1, 8, 8) if cond \
        else torch.zeros(len(x), NB, 8, 8, device=x.device)
    return torch.cat([x, oh], 1)

def legal_mask(legal):
    idx = legal.clone(); idx[idx < 0] = 4096
    m = torch.full((len(idx), 4097), float("-inf"), device=idx.device)
    m.scatter_(1, idx, 0.0)
    return m[:, :4096]

class ChessCNN(nn.Module):
    def __init__(self, cin=17 + NB, ch=64, nblocks=6, hch=32):
        super().__init__()
        def blk(i, o): return [nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU()]
        layers = blk(cin, ch)
        for _ in range(nblocks): layers += blk(ch, ch)
        self.body = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Conv2d(ch, hch, 1), nn.ReLU(), nn.Flatten(), nn.Linear(hch * 64, 4096))
    def forward(self, x): return self.head(self.body(x))

class LinearBaseline(nn.Module):
    """score(move) = a[from-square] + b[to-square], both linear in the flattened input."""
    def __init__(self, cin=17 + NB):
        super().__init__(); self.lin = nn.Linear(cin * 64, 128)
    def forward(self, x):
        z = self.lin(x.flatten(1))
        return (z[:, :64, None] + z[:, None, 64:]).flatten(1)

@torch.no_grad()
def evaluate(model, D, cond=True, shuffle_band=False, bs=2048):
    model.eval(); n = len(D["y"]); loss = 0.0; c1 = 0; c3 = 0
    per_c = np.zeros(NB); per_n = np.zeros(NB)
    for i in range(0, n, bs):
        sl = slice(i, i + bs)
        band = D["band"][sl]
        if shuffle_band: band = band[torch.randperm(len(band), device=band.device)]
        logits = model(make_input(D["X"][sl], band, cond)) + legal_mask(D["legal"][sl])
        y = D["y"][sl]
        loss += F.cross_entropy(logits, y, reduction="sum").item()
        top3 = logits.topk(3, 1).indices
        hit1 = (top3[:, 0] == y); hit3 = (top3 == y[:, None]).any(1)
        c1 += hit1.sum().item(); c3 += hit3.sum().item()
        for b in range(NB):
            m = D["band"][sl] == b
            per_c[b] += hit1[m].sum().item(); per_n[b] += m.sum().item()
    return dict(loss=loss / n, top1=c1 / n, top3=c3 / n, per_band_top1=(per_c / np.maximum(per_n, 1)).tolist())

def train(model, cond=True, epochs=EPOCHS, lr=1e-3, wd=1e-4, name="model"):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    hist = dict(train_loss=[], val_loss=[], val_top1=[]); best = (1e9, None)
    n = len(Dtr["y"])
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n, device=device); tot = 0.0; t0 = time.time()
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            logits = model(make_input(Dtr["X"][idx], Dtr["band"][idx], cond)) + legal_mask(Dtr["legal"][idx])
            loss = F.cross_entropy(logits, Dtr["y"][idx])
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
        v = evaluate(model, Dva, cond)
        hist["train_loss"].append(tot / n); hist["val_loss"].append(v["loss"]); hist["val_top1"].append(v["top1"])
        if v["loss"] < best[0]: best = (v["loss"], {k: t.clone() for k, t in model.state_dict().items()})
        print(f"[{name}] epoch {ep+1}/{epochs} train {tot/n:.3f} val {v['loss']:.3f} val top1 {v['top1']:.3f} ({time.time()-t0:.0f}s)")
    model.load_state_dict(best[1])      # early stopping: keep best validation loss
    return hist

## 4. Baselines

In [ ]:
# Baseline 1: hand-coded heuristic, no learning. Capture the most valuable piece (cheapest attacker on ties),
# otherwise play a uniformly random legal move.
VAL = np.array([1, 3, 3, 5, 9, 0])
def heuristic_eval(d, n_max=20000, seed=SEED):
    rng = np.random.RandomState(seed)
    idxs = rng.permutation(len(d["y"]))[:n_max]
    hits = np.zeros(NB); cnt = np.zeros(NB); n_cap = 0
    for i in idxs:
        x = d["X"][i]; legal = d["legal"][i]; legal = legal[legal >= 0]
        opp = sum(x[6 + k].reshape(64).astype(int) * VAL[k] for k in range(6))
        own = sum(x[k].reshape(64).astype(int) * VAL[k] for k in range(6))
        frm, to = legal // 64, legal % 64
        gain = opp[to]
        if gain.max() > 0:
            n_cap += 1
            cand = np.nonzero(gain == gain.max())[0]
            cand = cand[own[frm[cand]] == own[frm[cand]].min()]
            pick = legal[rng.choice(cand)]
        else:
            pick = legal[rng.randint(len(legal))]
        b = int(d["band"][i]); cnt[b] += 1; hits[b] += (pick == d["y"][i])
    return dict(top1=float(hits.sum() / cnt.sum()), per_band_top1=(hits / np.maximum(cnt, 1)).tolist(),
                capture_available=n_cap / len(idxs))

def random_legal_top1(d):
    return float((1.0 / (d["legal"] >= 0).sum(1)).mean())

heur = heuristic_eval(te)
rand_ref = random_legal_top1(te)
print("heuristic top1:", round(heur["top1"], 4), "| capture available in", round(heur["capture_available"], 3),
      "of positions | uniform-random-legal reference:", round(rand_ref, 4))

# Baseline 2: linear model (logistic-regression style) on the flattened input
lin = LinearBaseline(); lin_hist = train(lin, cond=True, name="linear")
lin_te = evaluate(lin, Dte); print("linear test:", lin_te["top1"], lin_te["top3"])

## 5. Primary model: CNN (conditioned on rating band) + ablation without band input

In [ ]:
cnn = ChessCNN(); n_params = sum(p.numel() for p in cnn.parameters())
print("CNN parameters:", f"{n_params:,}")
cnn_hist = train(cnn, cond=True, name="cnn")
cnn_te = evaluate(cnn, Dte)
cnn_shuf = evaluate(cnn, Dte, shuffle_band=True)
print("CNN test:", cnn_te["top1"], cnn_te["top3"], "| shuffled band input:", cnn_shuf["top1"])

cnn_nc = ChessCNN(); nc_hist = train(cnn_nc, cond=False, name="cnn-no-band")
nc_te = evaluate(cnn_nc, Dte, cond=False)
print("CNN w/o band test:", nc_te["top1"], nc_te["top3"])

## 6. Figures: learning curves, per-band accuracy, qualitative examples

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.1))
ep = range(1, len(cnn_hist["train_loss"]) + 1)
ax[0].plot(ep, cnn_hist["train_loss"], label="CNN train"); ax[0].plot(ep, cnn_hist["val_loss"], label="CNN val")
ax[0].plot(ep, lin_hist["val_loss"], "--", label="Linear val")
ax[0].set_xlabel("epoch", fontsize=8); ax[0].set_ylabel("cross-entropy", fontsize=8); ax[0].legend(fontsize=7)
ax[0].set_title("Learning curves", fontsize=9)
ax[1].plot(ep, cnn_hist["val_top1"], label="CNN"); ax[1].plot(ep, lin_hist["val_top1"], "--", label="Linear")
ax[1].plot(ep, nc_hist["val_top1"], ":", label="CNN, no band input")
ax[1].axhline(heur["top1"], color="gray", ls="-.", label="Capture heuristic"); ax[1].legend(fontsize=7)
ax[1].set_xlabel("epoch", fontsize=8); ax[1].set_ylabel("val top-1", fontsize=8); ax[1].set_title("Validation top-1 accuracy", fontsize=9)
w = .2; xs = np.arange(NB)
ax[2].bar(xs - 1.5 * w, heur["per_band_top1"], w, label="Heuristic", color="gray")
ax[2].bar(xs - .5 * w, lin_te["per_band_top1"], w, label="Linear")
ax[2].bar(xs + .5 * w, nc_te["per_band_top1"], w, label="CNN no band")
ax[2].bar(xs + 1.5 * w, cnn_te["per_band_top1"], w, label="CNN")
ax[2].set_xticks(xs); ax[2].set_xticklabels(BAND_NAMES, fontsize=7); ax[2].legend(fontsize=6)
ax[2].set_title("Test top-1 by rating band", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_curves.png", dpi=200); plt.close()

In [ ]:
@torch.no_grad()
def probs_for(model, D, i, band):
    model.eval()
    b = torch.tensor([band], device=device)
    lg = model(make_input(D["X"][i:i+1], b)) + legal_mask(D["legal"][i:i+1])
    return F.softmax(lg, 1)[0].cpu().numpy()

# pick the example where changing the band input changes the prediction the most (cherry-picked on purpose)
rng = np.random.RandomState(1); cand = rng.permutation(len(Dte["y"]))[:1500]
best_i, best_tv = 0, -1
for i in cand:
    p0 = probs_for(cnn, Dte, int(i), 0); p4 = probs_for(cnn, Dte, int(i), NB - 1)
    tv = 0.5 * np.abs(p0 - p4).sum()
    if tv > best_tv: best_i, best_tv = int(i), tv
# plus a typical confident-correct and a confident-wrong example
with torch.no_grad():
    cnn.eval(); lg = cnn(make_input(Dte["X"][:3000], Dte["band"][:3000])) + legal_mask(Dte["legal"][:3000])
    pr = F.softmax(lg, 1); conf, pred = pr.max(1); correct = pred == Dte["y"][:3000]
ok_i = int(torch.nonzero(correct & (conf > .6))[0]) if (correct & (conf > .6)).any() else 0
bad_i = int(torch.nonzero(~correct & (conf > .5))[0]) if (~correct & (conf > .5)).any() else 1

fig = plt.figure(figsize=(11, 6.2))
gs = fig.add_gridspec(2, 3)
for k, (i, title) in enumerate([(ok_i, "Example: model right"), (bad_i, "Example: model wrong")]):
    ax = fig.add_subplot(gs[0, k]); x = te["X"][i]; y = int(te["y"][i])
    p = probs_for(cnn, Dte, i, int(te["band"][i])); top = int(p.argmax())
    draw_board(ax, x, [(y, "#2ca02c", "-"), (top, "#d62728", "--")] if top != y else [(y, "#2ca02c", "-")])
    ax.set_title(f"{title}\nband {BAND_NAMES[int(te['band'][i])]}; green=played {mv_name(y)}, red=CNN {mv_name(top)} ({p[top]:.0%})", fontsize=8)
ax = fig.add_subplot(gs[0, 2]); ax.axis("off")
ax.text(0, .5, "Arrows: green = move the human\nplayed, red dashed = CNN top choice.\nMover is always at the bottom\n(boards are flipped for Black).\nBottom row: one position fed to the\nsame CNN with different band inputs.",
        fontsize=9, va="center")
x = te["X"][best_i]; y = int(te["y"][best_i])
ax = fig.add_subplot(gs[1, 0]); draw_board(ax, x, [(y, "#2ca02c", "-")])
ax.set_title(f"Largest band effect (selected from 1500 test positions)\nplayed {mv_name(y)}", fontsize=8)
P = np.stack([probs_for(cnn, Dte, best_i, b) for b in range(NB)])
topmoves = list(np.argsort(-P.max(0))[:4])
ax = fig.add_subplot(gs[1, 1:])
for j, mvi in enumerate(topmoves):
    ax.bar(np.arange(NB) + (j - 1.5) * .2, P[:, mvi], .2, label=mv_name(int(mvi)))
ax.set_xticks(range(NB)); ax.set_xticklabels(BAND_NAMES, fontsize=7); ax.set_ylabel("predicted probability", fontsize=8)
ax.set_xlabel("rating band given to the model", fontsize=8); ax.legend(fontsize=7, title="move", title_fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_qualitative.png", dpi=200); plt.close()
print("band effect TV distance on selected example:", round(best_tv, 3))

## 7. Final-test data: a completely different month (never used for training/validation/tuning)

In [ ]:
new_raw, new_stats = build_dataset(TEST_MONTH, TEST_GAMES_PER_BAND, SEED + 1)
Dnew = to_dev(new_raw)
cnn_new = evaluate(cnn, Dnew); lin_new = evaluate(lin, Dnew); nc_new = evaluate(cnn_nc, Dnew, cond=False)
heur_new = heuristic_eval(new_raw)
print("NEW MONTH | CNN", cnn_new["top1"], cnn_new["top3"], "| Linear", lin_new["top1"], "| Heuristic", heur_new["top1"])

## 8. Save everything

In [ ]:
res = dict(
    config=dict(train_month=TRAIN_MONTH, test_month=TEST_MONTH, games_per_band=GAMES_PER_BAND, pos_per_game=POS_PER_GAME,
                epochs=EPOCHS, batch=BATCH, device=str(device), synthetic=SYNTHETIC),
    data=dict(scanned=train_stats["scanned"], games_per_band=games_per_band, pos_per_band=pos_per_band,
              total_positions=int(len(train_raw["y"])), n_train=len(tr["y"]), n_val=len(va["y"]), n_test=len(te["y"]),
              avg_legal=avg_legal, new_month_positions=int(len(new_raw["y"])), new_month_games=new_stats["games_per_band"]),
    heuristic=heur, random_legal_top1=rand_ref, linear=lin_te, cnn=cnn_te, cnn_shuffled_band_top1=cnn_shuf["top1"],
    cnn_no_band=nc_te, cnn_params=n_params, linear_params=sum(p.numel() for p in lin.parameters()),
    new_month=dict(cnn=cnn_new, linear=lin_new, cnn_no_band=nc_new, heuristic=heur_new),
    history=dict(cnn=cnn_hist, linear=lin_hist, cnn_no_band=nc_hist), band_effect_tv=float(best_tv))
best_tv = float(best_tv)
json.dump(res, open(f"{OUT}/results.json", "w"), indent=1, default=float)

def pct(v): return f"{100*v:.1f}"
with open(f"{OUT}/numbers.tex", "w") as f:
    def mac(name, val): f.write(f"\\newcommand{{\\{name}}}{{{val}}}\n")
    mac("nTotal", f"{len(train_raw['y']):,}".replace(",", "{,}")); mac("nTrain", f"{len(tr['y']):,}".replace(",", "{,}"))
    mac("nVal", f"{len(va['y']):,}".replace(",", "{,}")); mac("nTest", f"{len(te['y']):,}".replace(",", "{,}"))
    mac("nNew", f"{len(new_raw['y']):,}".replace(",", "{,}")); mac("nGamesBand", f"{min(games_per_band):,}".replace(",", "{,}"))
    mac("nScanned", f"{train_stats['scanned']:,}".replace(",", "{,}")); mac("avgLegal", f"{avg_legal:.0f}")
    mac("cnnParams", f"{n_params/1e6:.1f}M")
    mac("linParams", f"{sum(p.numel() for p in lin.parameters()):,}".replace(",", "{,}"))
    for key, r in [("Cnn", cnn_te), ("Lin", lin_te), ("Nc", nc_te)]:
        mac(f"topOne{key}", pct(r["top1"])); mac(f"topThree{key}", pct(r["top3"])); mac(f"loss{key}", f"{r['loss']:.2f}")
    mac("topOneHeur", pct(heur["top1"])); mac("topOneRand", pct(rand_ref)); mac("capAvail", pct(heur["capture_available"]))
    mac("topOneCnnShuf", pct(cnn_shuf["top1"]))
    mac("newCnn", pct(cnn_new["top1"])); mac("newCnnThree", pct(cnn_new["top3"]))
    mac("newLin", pct(lin_new["top1"])); mac("newHeur", pct(heur_new["top1"])); mac("newNc", pct(nc_new["top1"]))
    mac("bandTV", f"{best_tv:.2f}")
print(open(f"{OUT}/numbers.tex").read())

shutil.make_archive("progress_outputs", "zip", OUT)
print("\n==== SEND progress_outputs.zip BACK (or paste results.json) ====")
print(json.dumps({k: res[k] for k in ["data", "heuristic", "linear", "cnn", "cnn_no_band", "new_month", "cnn_params", "cnn_shuffled_band_top1"]}, indent=1)[:3500])
try:
    from google.colab import files; files.download("progress_outputs.zip")
except Exception:
    pass